In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"), override=True)

%load_ext autoreload
%autoreload 2

## 规划：TODO 列表

![Screenshot 2025-08-15 at 11.41.46 AM.png](attachment:e94d5803-050b-4502-b784-f520a96c322b.png)

许多智能体使用 TODO 列表作为关键导航工具，帮助引导完成长期运行的复杂任务。Claude Code 利用[计划模式](https://www.anthropic.com/engineering/claude-code-best-practices)在执行任务前创建结构化的 TODO 列表，使用基于 [Claude Code 提示](https://cchistory.mariozechner.at/)的特定工具 `TodoWrite`。每个 TODO 项目包含两个关键组件：内容（简短、具体的任务描述）和状态（待处理、进行中或已完成）。

TODO 列表的挑战在于随着上下文窗口增长而保持注意力——平均 Manus 任务使用约50次工具调用，这会产生[上下文腐烂](https://research.trychroma.com/context-rot)的巨大风险。智能体在冗长对话或复杂任务期间容易偏离主题或忘记早期目标。通过持续重写和更新 TODO 列表，像 Manus 这样的智能体有效地在上下文末尾重述其目标，帮助[保持任务专注](https://manus.im/blog/Context-Engineering-for-AI-Agents-Lessons-from-Building-Manus)并防止任务偏移。

<!-- 下面的样式减少了同一项目符号列表中项目之间的间隙。每个笔记本运行一次 -->
<style>
/* JupyterLab + classic notebook */
.jp-RenderedHTMLCommon ul, .text_cell_render ul { margin-top: .25em; margin-bottom: .35em; padding-left: 1.2em; }
.jp-RenderedHTMLCommon ul ul, .text_cell_render ul ul { margin-top: .15em; margin-bottom: .15em; padding-left: 1.0em; }
.jp-RenderedHTMLCommon li, .text_cell_render li { margin: .1em 0; }
</style>

### 状态

就像上一课一样，您将使用带有自定义状态的 `create_react_agent`。

状态对象作为我们在工作流程不同阶段之间存储和传递上下文的主要机制。状态由图的模式以及指定如何应用状态更新的 reducer 函数组成。

DeepAgent 模式中定义了三个主要元素：**`messages`**、**`todo`** 和 **`files`**。

- **`messages`** 继承自 `AgentState`，在第一课中已经描述过。
  - `add_messages` reducer 将新消息追加到消息列表的末尾。
- **`todo`** 是 `Todo` 任务的列表。每个任务都有一个描述：`content` 和一个 `status`：待处理、进行中、已完成。
  - 没有定义自定义 reducer，因此更新会在写入时覆盖列表。
- **`files`** 是包含在状态中的虚拟文件系统，您将在下一课中探索。

In [ ]:
%%writefile ../src/deep_agents_from_scratch/state.py
"""深度智能体的状态管理，支持 TODO 跟踪和虚拟文件系统。

此模块定义了支持以下功能的扩展智能体状态结构：
- 通过 TODO 列表进行任务规划和进度跟踪
- 通过存储在状态中的虚拟文件系统进行上下文卸载
- 使用 reducer 函数进行高效状态合并
"""

from typing import Annotated, Literal, NotRequired
from typing_extensions import TypedDict

from langgraph.prebuilt.chat_agent_executor import AgentState


class Todo(TypedDict):
    """用于跟踪复杂工作流程进度的结构化任务项目。

    属性:
        content: 任务的简短、具体描述
        status: 当前状态 - 待处理、进行中或已完成
    """

    content: str
    status: Literal["pending", "in_progress", "completed"]


def file_reducer(left, right):
    """合并两个文件字典，右侧优先。

    用作智能体状态中文件字段的 reducer 函数，
    允许对虚拟文件系统进行增量更新。

    Args:
        left: 左侧字典（现有文件）
        right: 右侧字典（新/更新的文件）

    Returns:
        合并的字典，右侧值覆盖左侧值
    """
    if left is None:
        return right
    elif right is None:
        return left
    else:
        return {**left, **right}


class DeepAgentState(AgentState):
    """包含任务跟踪和虚拟文件系统的扩展智能体状态。

    继承自 LangGraph 的 AgentState 并添加：
    - todos: 用于任务规划和进度跟踪的 Todo 项目列表
    - files: 存储为将文件名映射到内容的字典的虚拟文件系统
    """

    todos: NotRequired[list[Todo]]
    files: Annotated[NotRequired[dict[str, str]], file_reducer]

### 工具描述 - Todo 工具
如上所述，长期运行的智能体可以使用 todo 列表来保持任务专注。为了实现这一点，创建了 todo 工具，`write_todo` 和 `read_todo`。下面的工具描述提供给 LLM，详细说明何时使用 todo 列表、它包含什么以及如何读取和更新它。
注意，虽然列表包含单个任务，但它是通过完全重写列表来更新的。这允许 LLM 在取得进展时重新考虑任务。

In [ ]:
from utils import show_prompt

from deep_agents_from_scratch.prompts import WRITE_TODOS_DESCRIPTION

show_prompt(WRITE_TODOS_DESCRIPTION)

### 写入和读取 ToDo 工具

下面，您将创建 **`write_todos`** 和 **`read_todos`** 工具：
**`write_todos`** 工具从 LLM 接收 todo 列表作为参数，并将其写入状态，覆盖任何先前的列表。然后它返回一个包含已写入列表的 `ToolMessage`。注意，写入列表使信息在存储在 `messages` 中的对话历史中对 LLM 可用，无论是在 LLM 生成的工具调用中还是在返回的 `ToolMessage` 中。

**`read_todos`** 工具从状态读取 todo 列表并将其作为 `ToolMessage` 返回。它可以用于在 LLM 上下文中刷新信息。

注意，这些工具将使用上一课讨论的一些功能：

- `InjectedState` 为工具提供对图状态的访问。
- `Command` 更新状态中的值。

In [ ]:
%%writefile ../src/deep_agents_from_scratch/todo_tools.py
"""用于任务规划和进度跟踪的 TODO 管理工具。

此模块提供用于创建和管理结构化任务列表的工具，
使智能体能够规划复杂工作流程并跟踪
多步操作的进度。
"""

from typing import Annotated

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command

from deep_agents_from_scratch.prompts import WRITE_TODOS_DESCRIPTION
from deep_agents_from_scratch.state import DeepAgentState, Todo


@tool(description=WRITE_TODOS_DESCRIPTION,parse_docstring=True)
def write_todos(
    todos: list[Todo], tool_call_id: Annotated[str, InjectedToolCallId]
) -> Command:
    """创建或更新智能体的 TODO 列表以进行任务规划和跟踪。

    Args:
        todos: 包含内容和状态的 Todo 项目列表
        tool_call_id: 消息响应的工具调用标识符

    Returns:
        用新 TODO 列表更新智能体状态的命令
    """
    return Command(
        update={
            "todos": todos,
            "messages": [
                ToolMessage(f"将 todo 列表更新为 {todos}", tool_call_id=tool_call_id)
            ],
        }
    )


@tool(parse_docstring=True)
def read_todos(
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> str:
    """从智能体状态读取当前 TODO 列表。

    此工具允许智能体检索和查看当前 TODO 列表
    以保持对剩余任务的专注并跟踪复杂工作流程的进度。

    Args:
        state: 包含当前 TODO 列表的注入智能体状态
        tool_call_id: 用于消息跟踪的注入工具调用标识符

    Returns:
        当前 TODO 列表的格式化字符串表示
    """
    todos = state.get("todos", [])
    if not todos:
        return "当前列表中没有 todos。"

    result = "当前 TODO 列表：\n"
    for i, todo in enumerate(todos, 1):
        status_emoji = {"pending": "⏳", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result += f"{i}. {emoji} {todo['content']} ({todo['status']})\n"

    return result.strip()

### 图

如第一课所示，您将使用 `create_react_agent` 构建智能体。我们将专注于让我们的智能体使用 todo 列表。我们将遵循 Manus 的方法，在每个任务后进行 todo 重述，并使用您刚刚在上面创建的工具。

In [ ]:
from deep_agents_from_scratch.prompts import TODO_USAGE_INSTRUCTIONS

show_prompt(TODO_USAGE_INSTRUCTIONS)

In [ ]:
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from utils import format_messages

from deep_agents_from_scratch.prompts import TODO_USAGE_INSTRUCTIONS
from deep_agents_from_scratch.state import DeepAgentState
from deep_agents_from_scratch.todo_tools import read_todos, write_todos

# 模拟搜索结果
search_result = """模型上下文协议（MCP）是由 Anthropic 开发的开放标准协议，
用于实现 AI 模型与外部系统（如工具、数据库和其他服务）之间的无缝集成。
它充当标准化通信层，允许 AI 模型以一致和高效的方式访问和利用来自各种来源的数据。
本质上，MCP 通过为数据交换提供统一语言来简化将 AI 助手连接到外部服务的过程。"""


# 模拟搜索工具
@tool(parse_docstring=True)
def web_search(
    query: str,
):
    """搜索网络以获取特定主题的信息。

    此工具执行网络搜索并返回给定查询的相关结果。
    当您需要从互联网收集有关任何主题的信息时使用此工具。

    Args:
        query: 搜索查询字符串。要具体明确您要查找的信息。

    Returns:
        来自搜索引擎的搜索结果。

    Example:
        web_search("机器学习在医疗保健中的应用")
    """
    return search_result


# 直接使用 create_react_agent 创建智能体
model = init_chat_model(model="anthropic:claude-sonnet-4-20250514", temperature=0.0)
tools = [write_todos, web_search, read_todos]

# 添加模拟研究指令
SIMPLE_RESEARCH_INSTRUCTIONS = """重要：只需调用一次 web_search 工具，并使用工具提供的结果来回答用户的问题。"""

# 创建智能体
agent = create_react_agent(
    model,
    tools,
    prompt=TODO_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + SIMPLE_RESEARCH_INSTRUCTIONS,
    state_schema=DeepAgentState,
)

# 显示智能体
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

启动图，状态中没有 `todos`，并发出用户研究请求。

In [ ]:
# 使用示例
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "给我一个模型上下文协议（MCP）的简短摘要。",
            }
        ],
        "todos": [],
    }
)

format_messages(result["messages"])

跟踪：
https://smith.langchain.com/public/7e771e79-8996-4833-8242-2b3559c2c1d7/r
<!-- https://smith.langchain.com/public/57e530e5-c3bb-4ed6-aff6-c57d06158fe9/r  -->